# Asistente turístico de Tenerife

Asistente conversacional que ayuda a turistas a planificar su viaje a la isla de
Tenerife. Combina tres capacidades sobre **Google Gemini** (vía LangChain):

1. **RAG** sobre la guía oficial `TENERIFE.pdf`, expuesto como una **herramienta**
   (`search_tourist_guide`) que el modelo decide invocar.
2. Una **function call** externa `get_weather` que consulta la previsión
   meteorológica real (Open-Meteo) con respaldo simulado.
3. **Diálogo multiturno** con memoria de contexto y respuesta en *streaming*.

## Objetivos del enunciado

- Construir un pipeline **RAG** (carga de PDF, troceado, embeddings, FAISS) y
  exponerlo como herramienta con **citas de fuente** (página y fragmento).
- Integrar una **herramienta externa** mediante *function calling* (`get_weather`).
- Mantener una **conversación multiturno** con gestión del historial.
- Evaluar el comportamiento del asistente con un conjunto reproducible de prompts.

## Arquitectura

```
Usuario ─▶ TouristAssistant (Gemini + bind_tools)
                │  bucle de tool calling
                ├─▶ search_tourist_guide ─▶ TouristGuideRAG (FAISS) ─▶ TENERIFE.pdf
                └─▶ get_weather ─────────▶ Open-Meteo (fallback simulado)
                │
                └─▶ respuesta final en español (streaming)
```

El modelo recibe el esquema JSON de cada herramienta (derivado de los *type hints*
y el *docstring*) y **elige autónomamente** cuál llamar en cada turno. El historial
se recorta con `trim_messages` para controlar la longitud.

## Resumen de decisiones técnicas

| Decisión | Valor | Motivo |
|---|---|---|
| Modelo de generación | `gemini-2.5-flash-lite` | Rápido y económico para diálogo |
| Embeddings | `models/gemini-embedding-001` | Coherencia con el proveedor |
| Vector store | **FAISS** persistente en disco | Indexado una sola vez |
| Troceado | `chunk_size=1000`, `overlap=150` | Equilibrio contexto/precisión |
| `top_k` | 4 | Suficiente contexto sin ruido |
| Temperatura | 0.2 | Respuestas fieles a la guía |
| RAG como herramienta | `@tool` | El modelo decide cuándo recuperar |
| Respaldo meteorológico | simulación determinista | El asistente nunca se queda sin respuesta |

> El código (identificadores) está en **inglés**; la documentación y los textos
> visibles para el turista, en **español**.


## 1. Configuración y credenciales

Cargamos la configuración inmutable con `load_settings()`. Esta función lee el
archivo `.env` (vía `load_dotenv`), deja `GOOGLE_API_KEY` en el entorno para que
`langchain-google-genai` la detecte y construye un objeto `Settings` con los
parámetros del modelo y del pipeline RAG.

La siguiente celda ajusta el directorio de trabajo a la raíz del proyecto para
poder importar el paquete `core`.


In [ ]:
import os
import sys
from pathlib import Path

# Aseguramos que la raíz del proyecto está en sys.path para importar "core".
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "core").exists():
    # Si el notebook se ejecuta desde otra carpeta, ajusta esta ruta.
    PROJECT_ROOT = Path(
        "/home/manu/dev/github.com/manupm87/pontia-llm/project"
    )
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Directorio de trabajo:", Path.cwd())

In [ ]:
from core.config import load_settings

settings = load_settings()

# Parámetros del modelo de generación.
print("Modelo de generación:", settings.generation_model)
print("Modelo de embeddings:", settings.embedding_model)
print("temperature        :", settings.temperature)
print("top_p              :", settings.top_p)
print("max_output_tokens  :", settings.max_output_tokens)
print("top_k (RAG)        :", settings.top_k)
print("chunk_size/overlap :", settings.chunk_size, "/", settings.chunk_overlap)
print()
print("¿Hay clave de API configurada? ->", settings.has_api_key)

if not settings.has_api_key:
    print(
        "\nAVISO: define GOOGLE_API_KEY en un archivo .env para ejecutar "
        "las celdas que llaman a Gemini."
    )


## 2. RAG sobre la guía (TENERIFE.pdf)

El motor `TouristGuideRAG` implementa el pipeline de recuperación:

1. **Carga** del PDF con `PyPDFLoader`.
2. **Troceado** con `RecursiveCharacterTextSplitter` (`chunk_size=1000`,
   `chunk_overlap=150`, `add_start_index=True`). A cada fragmento se le añaden
   metadatos de cita: `source_name` y `chunk_id`.
3. **Embeddings** con Gemini (`models/gemini-embedding-001`).
4. **Índice FAISS** persistente: se construye una vez y se recarga desde disco en
   ejecuciones posteriores (`build_index`).
5. **Imágenes**: `GuideImageStore` (en `core/images.py`) extrae las fotos del PDF
   con PyMuPDF y las indexa por página. `build_index` las prepara junto al índice.

`retrieve(query)` devuelve el **contexto formateado** (con encabezados de cita
`[Fuente i: archivo, página P, fragmento C]`), la lista de **fuentes** citadas y
las **fotos** de las páginas recuperadas, que también quedan en `last_sources` y
`last_images` para la interfaz.

In [ ]:
from core.rag import TouristGuideRAG

rag = TouristGuideRAG(settings)
# Construye el índice FAISS y extrae las fotos del PDF (o lo recarga si ya
# existen en storage/).
rag.build_index()

# Inspeccionamos cuántos fragmentos contiene el índice y cuántas fotos hay.
num_chunks = rag.vector_store.index.ntotal
print(f"Índice FAISS listo con {num_chunks} fragmentos.")
print(f"Fotos de la guía extraídas: {rag.image_store.total_images}")
print("PDF indexado:", settings.pdf_path.name)

In [ ]:
# Demostración de recuperación con citas.
query = "¿Qué playas de arena negra puedo visitar en Tenerife?"
result = rag.retrieve(query)

print("CONSULTA:", query)
print("=" * 70)
print(result["context"][:1200], "...")
print("=" * 70)
print("\nFUENTES CITADAS:")
for i, src in enumerate(result["sources"], start=1):
    page = src["page"]
    page_label = page + 1 if isinstance(page, int) else "?"
    print(
        f"  [{i}] {src['source_name']} | página {page_label} | "
        f"fragmento {src['chunk_id']}"
    )
    print(f"      {src['snippet']!r}")

# La recuperación también devuelve las fotos de las páginas recuperadas, que la
# interfaz muestra junto a la respuesta.
print("\nFOTOS DE LA GUÍA (lugares recuperados):")
for img in result["images"]:
    print(f"  - {img['caption'] or 'Foto'} (página {img['page'] + 1})")

# Vista previa de las fotos dentro del propio notebook.
from IPython.display import Image, display

for img in result["images"]:
    display(Image(filename=img["path"], width=360))

## 3. Herramienta meteorológica externa `get_weather`

`get_weather(date)` consulta la API pública **Open-Meteo** para la previsión
diaria de Tenerife. Características:

- Valida el formato `YYYY-MM-DD`; si es inválido, lanza `ValueError` (que la
  herramienta del LLM captura para avisar al usuario).
- Ante **cualquier fallo de red o parseo**, recurre a una **simulación
  determinista** (`source = "simulada (fallback)"`), de modo que el asistente
  siempre dispone de una respuesta.
- Cada intento se registra en `WEATHER_CALL_LOG` (fecha, éxito, fuente, tiempo,
  error) para observabilidad.

Open-Meteo solo ofrece previsión a corto plazo, así que una **fecha lejana**
provoca el fallback simulado de forma natural.


In [ ]:
from datetime import date as _date, timedelta

from core.weather import get_weather, WEATHER_CALL_LOG

# Fecha cercana: debería resolverse con datos reales de Open-Meteo.
near_date = (_date.today() + timedelta(days=2)).isoformat()
# Fecha lejana: fuera del horizonte de previsión -> fallback simulado.
far_date = (_date.today() + timedelta(days=365)).isoformat()

near = get_weather(near_date)
far = get_weather(far_date)

print(f"Fecha cercana ({near_date}):")
print("  ", near)
print(f"\nFecha lejana ({far_date}):")
print("  ", far)


In [ ]:
# Registro de observabilidad de las llamadas meteorológicas.
print("WEATHER_CALL_LOG:")
for entry in WEATHER_CALL_LOG:
    print(
        f"  {entry['date']} | ok={entry['ok']} | {entry['source']} | "
        f"{entry['elapsed_s']:.3f}s | error={entry['error']}"
    )


## 4. Definición de herramientas para el LLM

Las herramientas se definen con el decorador `@tool` de LangChain. El **nombre**
que ve el modelo es el de la función y el **esquema JSON** de los argumentos se
deriva automáticamente de los *type hints* y el *docstring* en español.

`get_tools()` devuelve la lista `[search_tourist_guide, get_weather]`. Antes de
usar la búsqueda hay que inyectar la instancia RAG con `set_rag_instance`.


In [ ]:
from core.tools import get_tools, set_rag_instance

# Inyectamos la instancia RAG compartida que usará search_tourist_guide.
set_rag_instance(rag)

tools = get_tools()
for t in tools:
    print("Herramienta :", t.name)
    print("Descripción :", t.description.strip().splitlines()[0])
    print("Args (schema):", t.args)
    print("-" * 70)


## 5. El asistente conversacional (tool calling + multiturno + streaming)

`TouristAssistant` orquesta el diálogo:

- En el constructor inyecta el RAG (`set_rag_instance`), inicializa el LLM con
  `init_chat_model(..., model_provider="google_genai")`, enlaza las herramientas
  con `bind_tools` y arranca el historial con el `SYSTEM_PROMPT` en español. El
  prompt obliga a consultar la guía en cada pregunta turística y a responder
  **solo** con lo recuperado (*grounding*).
- `chat()` / `stream()` ejecutan un **bucle de tool calling** sobre una **copia
  efímera** del historial: invocan el modelo con herramientas; si pide alguna, la
  ejecuta y repite; cuando responde sin pedir herramientas, ese texto es la
  respuesta final. En la **memoria** solo persisten los turnos de usuario y
  asistente (no las peticiones ni los resultados de herramientas), de modo que la
  guía se vuelve a consultar en cada turno y se refrescan **fuentes e imágenes**.
  En `stream()`, la respuesta final se genera token a token con el LLM **sin**
  herramientas.
- Cada llamada a herramienta se registra en `tool_log` (`ToolCallRecord`).
- `_trim_history()` recorta el historial con `trim_messages` para controlar la
  longitud (`max_history_messages`), conservando siempre el mensaje de sistema.

In [ ]:
from core.assistant import TouristAssistant, SYSTEM_PROMPT

assistant = TouristAssistant(settings, rag)

print("Herramientas enlazadas:", list(assistant.tools_by_name.keys()))
print("Mensajes iniciales en el historial:", len(assistant.history))
print()
print("SYSTEM_PROMPT:")
print(SYSTEM_PROMPT)


## 6. Conversación de ejemplo multiturno

Encadenamos varios turnos para demostrar:

- **Memoria de contexto** (el segundo turno se apoya en el primero sin repetir el
  tema).
- **Al menos 3 invocaciones de herramientas**: varias búsquedas en la guía y una
  consulta meteorológica.

Usamos `chat()` (respuesta completa) para poder inspeccionar fuentes y trazas en
cada turno.


In [ ]:
def show_turn(turn_number, user_message, response):
    """Imprime de forma legible un turno de la conversación."""
    print(f"\n{'#' * 70}")
    print(f"TURNO {turn_number} | Usuario: {user_message}")
    print("#" * 70)
    print("Asistente:", response["answer"])
    if response["tool_calls"]:
        print("\nHerramientas usadas en este turno:")
        for record in response["tool_calls"]:
            print(
                f"  - {record.name}({record.arguments}) "
                f"ok={record.ok} {record.elapsed_s:.2f}s"
            )
    if response["sources"]:
        print("\nFuentes citadas:")
        for src in response["sources"]:
            page = src["page"]
            page_label = page + 1 if isinstance(page, int) else "?"
            print(f"  - {src['source_name']} (página {page_label})")


In [ ]:
from datetime import date as _date, timedelta

trip_date = (_date.today() + timedelta(days=3)).isoformat()

conversation = [
    "Hola, ¿qué lugares imprescindibles puedo visitar en el Teide?",
    "Genial. ¿Y qué platos típicos canarios debería probar durante el viaje?",
    f"Quiero subir al Teide el {trip_date}. ¿Qué tiempo hará ese día?",
]

for i, message in enumerate(conversation, start=1):
    response = assistant.chat(message)
    show_turn(i, message, response)


In [ ]:
# Registro global de herramientas a lo largo de toda la conversación.
print("assistant.tool_log (todas las llamadas):")
for record in assistant.tool_log:
    print(
        f"  {record.name:<22} args={record.arguments} "
        f"ok={record.ok} {record.elapsed_s:.2f}s"
    )

num_guide = sum(1 for r in assistant.tool_log if r.name == "search_tourist_guide")
num_weather = sum(1 for r in assistant.tool_log if r.name == "get_weather")
print(f"\nBúsquedas en la guía: {num_guide} | Consultas meteorológicas: {num_weather}")
print("Total de invocaciones de herramientas:", len(assistant.tool_log))


## 7. Pruebas y casos límite

Comprobamos tres situaciones delicadas:

1. **Pregunta fuera de la guía**: el asistente debe reconocer que no dispone de la
   información y no inventar.
2. **Fecha inválida** en `get_weather`: la herramienta captura el `ValueError` y
   devuelve un mensaje de error en español.
3. **Fecha lejana**: la previsión recurre al **fallback simulado**.


In [ ]:
# Caso 1: pregunta claramente fuera del dominio de la guía de Tenerife.
assistant.reset()
out_of_scope = assistant.chat(
    "¿Cuál es el precio actual de las acciones de Apple en bolsa?"
)
print("Pregunta fuera de la guía:")
print(out_of_scope["answer"])


In [ ]:
# Caso 2: fecha con formato inválido -> la herramienta devuelve error controlado.
from core.tools import get_weather as weather_tool

invalid = weather_tool.invoke({"date": "14-06-2026"})
print("Resultado con fecha inválida:")
print(" ", invalid)


In [ ]:
# Caso 3: fecha lejana -> fallback simulado determinista.
from core.weather import get_weather

far = get_weather("2030-08-15")
print("Previsión para fecha lejana (fallback simulado):")
print(" ", far)
assert far["source"] == "simulada (fallback)", "Debería usar la simulación de respaldo"
print("\nOK: se usó la previsión simulada de respaldo.")


## 8. Evaluación

Definimos un conjunto **reproducible** de prompts y evaluamos heurísticas
sencillas sobre el comportamiento del asistente:

- **¿Selecciona la herramienta correcta?** (la esperada según el tipo de pregunta).
- **¿Cita alguna fuente?** (relevante para las preguntas sobre la guía).

Construimos un `DataFrame` con los resultados y una gráfica de barras. Estas son
heurísticas orientativas, no una evaluación semántica de la calidad de la
respuesta (ver limitaciones).


In [ ]:
# Conjunto de evaluación: (pregunta, herramienta esperada).
EVAL_PROMPTS = [
    ("¿Qué puedo ver en el Parque Nacional del Teide?", "search_tourist_guide"),
    ("Recomiéndame playas tranquilas en el sur de Tenerife.", "search_tourist_guide"),
    ("¿Qué tiempo hará en Tenerife el 2026-06-20?", "get_weather"),
    ("¿Dónde puedo probar buena gastronomía canaria?", "search_tourist_guide"),
    ("¿Lloverá en Tenerife el 2026-07-01?", "get_weather"),
]

eval_rows = []
for question, expected_tool in EVAL_PROMPTS:
    assistant.reset()
    response = assistant.chat(question)
    used_tools = [r.name for r in response["tool_calls"]]
    eval_rows.append(
        {
            "pregunta": question,
            "herramienta_esperada": expected_tool,
            "herramientas_usadas": ", ".join(used_tools) or "(ninguna)",
            "herramienta_correcta": expected_tool in used_tools,
            "cita_fuente": len(response["sources"]) > 0,
        }
    )

eval_rows


In [ ]:
import pandas as pd

df = pd.DataFrame(eval_rows)
pd.set_option("display.max_colwidth", 50)
df


In [ ]:
# Métricas agregadas.
accuracy_tool = df["herramienta_correcta"].mean()
# La cita de fuente solo se espera en preguntas sobre la guía.
guide_mask = df["herramienta_esperada"] == "search_tourist_guide"
citation_rate = df.loc[guide_mask, "cita_fuente"].mean()

print(f"Acierto de selección de herramienta: {accuracy_tool:.0%}")
print(f"Tasa de cita de fuente (preguntas de guía): {citation_rate:.0%}")


In [ ]:
import matplotlib.pyplot as plt

metrics = {
    "Herramienta\ncorrecta": accuracy_tool,
    "Cita fuente\n(guía)": citation_rate,
}

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(list(metrics.keys()), list(metrics.values()), color=["#2a9d8f", "#e76f51"])
ax.set_ylim(0, 1)
ax.set_ylabel("Proporción")
ax.set_title("Evaluación heurística del asistente")
for i, value in enumerate(metrics.values()):
    ax.text(i, value + 0.02, f"{value:.0%}", ha="center")
plt.tight_layout()
plt.show()


**Limitaciones de esta evaluación.** Son heurísticas estructurales: comprueban
*qué* herramienta se usa y *si* hay cita, pero no la **veracidad** ni la
**calidad** de la respuesta. Una evaluación más robusta requeriría un conjunto de
referencia anotado, *LLM-as-judge* para fidelidad al contexto recuperado y
métricas RAG específicas (precisión/cobertura de recuperación). Además, las
respuestas de Gemini no son deterministas, por lo que los resultados pueden variar
ligeramente entre ejecuciones.


## 9. Conclusiones, limitaciones y mejoras futuras

### Conclusiones

- Hemos construido un asistente turístico **multiturno** sobre Gemini que combina
  **RAG con citas** (expuesto como herramienta) y una **function call externa**
  (`get_weather`), con el modelo decidiendo autónomamente cuándo invocarlas.
- El diseño en el paquete `core/` separa responsabilidades (config, RAG, weather,
  tools, assistant) y se reutiliza tanto en este notebook como en la app Streamlit.
- El índice FAISS persistente y el respaldo meteorológico simulado hacen el sistema
  **robusto y reproducible**.

### Limitaciones

- La calidad de las respuestas depende de la cobertura del PDF y del troceado.
- Open-Meteo solo cubre un horizonte temporal limitado (fechas lejanas usan la
  simulación de respaldo).
- La evaluación es heurística (ver sección 8).

### Mejoras futuras

- *Re-ranking* de fragmentos y *chunking* semántico para mejorar la recuperación.
- Evaluación automática de fidelidad (RAGAS / LLM-as-judge).
- Más herramientas (mapas, rutas, eventos, reservas).

### Cómo ejecutar la aplicación

La interfaz interactiva está en `app.py` (Streamlit). Desde la raíz del proyecto:

```bash
streamlit run app.py
```

Necesitas un archivo `.env` con `GOOGLE_API_KEY=...`. Consulta también
**`README.md`** (instalación y uso) e **`INFORME.md`** (memoria técnica del
proyecto) para más detalles.
